Import libraries

In [1]:
import json
import re
from pathlib import Path

print("Libraries imported successfully!")

Libraries imported successfully!


Load the extracted documents

In [2]:
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "knowledge_base"
    / "processed"
    / "extracted_documents.jsonl"
)

documents = []

with open(INPUT_FILE, "r", encoding="utf-8") as file:
    for line in file:
        documents.append(json.loads(line))

print("Total extracted pages:", len(documents))

Total extracted pages: 321


Explore the raw text

In [3]:
sample = documents[0]

print("Source:", sample["source"])
print("Page:", sample["page"])
print("\nRaw text:\n")
print(sample["text"][:2000])

Source: omguide_complete_w-eo-disclaimer.pdf
Page: 1

Raw text:

FEDERAL ENERGY MANAGEMENT PROGRAM 
Release 3.0 
Operations & Maintenance 
Best Practices 
A Guide to Achieving Operational Efficiency 
August 2010 



Create a text-cleaning function

In [4]:
def clean_text(text):

    # Normalize line endings
    text = text.replace("\r\n", "\n")

    # Remove null characters
    text = text.replace("\x00", "")

    # Join words split by a hyphen at a line break
    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)

    # Replace multiple spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    # Reduce excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove unnecessary spaces around line breaks
    text = re.sub(r" *\n *", "\n", text)

    return text.strip()

Clean all extracted documents

In [5]:
cleaned_documents = []

for document in documents:

    cleaned_text = clean_text(document["text"])

    if cleaned_text:

        cleaned_documents.append({
            "source": document["source"],
            "page": document["page"],
            "text": cleaned_text
        })

print("Original pages:", len(documents))
print("Cleaned pages:", len(cleaned_documents))
print("Cleaning completed!")

Original pages: 321
Cleaned pages: 307
Cleaning completed!


Compare raw and cleaned text

In [6]:
index = 0

print("========== RAW TEXT ==========")
print(documents[index]["text"][:1000])

print("\n========== CLEANED TEXT ==========")
print(cleaned_documents[index]["text"][:1000])

========== RAW TEXT ==========
FEDERAL ENERGY MANAGEMENT PROGRAM 
Release 3.0 
Operations & Maintenance 
Best Practices 
A Guide to Achieving Operational Efficiency 
August 2010 


========== CLEANED TEXT ==========
FEDERAL ENERGY MANAGEMENT PROGRAM
Release 3.0
Operations & Maintenance
Best Practices
A Guide to Achieving Operational Efficiency
August 2010


Save cleaned documents

In [7]:
OUTPUT_FILE = (
    PROJECT_ROOT
    / "knowledge_base"
    / "processed"
    / "cleaned_documents.jsonl"
)

with open(OUTPUT_FILE, "w", encoding="utf-8") as file:

    for document in cleaned_documents:
        file.write(
            json.dumps(document, ensure_ascii=False) + "\n"
        )

print("Cleaned documents saved:", OUTPUT_FILE)

Cleaned documents saved: d:\FactoryMindAI\knowledge_base\processed\cleaned_documents.jsonl


Validate the cleaning results

In [8]:
print("Total cleaned pages:", len(cleaned_documents))

print(
    "Total characters:",
    sum(len(doc["text"]) for doc in cleaned_documents)
)

print(
    "Empty documents:",
    sum(not doc["text"].strip() for doc in cleaned_documents)
)

Total cleaned pages: 307
Total characters: 774428
Empty documents: 0
